# A3-M1 · 05 토큰으로 유사도
토크나이저 다섯 가지로 TF-IDF 검색을 만들어 Hit@1·Hit@3을 비교합니다. 02 노트북을 먼저 실행해야 합니다.

## 12-1. 준비 — FAQ와 평가 질문, 토크나이저 다섯 개

In [1]:
import re, numpy as np, tiktoken
from kiwipiepy import Kiwi
from tokenizers import Tokenizer
from shop_faq import FAQ, TESTS

kiwi = Kiwi()
wp = Tokenizer.from_file("shop_wordpiece.json")         # 02 노트북에서 만든 파일
o200k = tiktoken.get_encoding("o200k_base")
KEEP = {"NNG", "NNP", "NNB", "SL", "SN", "VV", "VA", "XR"}

TOKENIZERS = {
    "띄어쓰기":   lambda s: s.split(),
    "글자 2-gram": lambda s: [s[i:i+2] for i in range(len(s) - 1) if " " not in s[i:i+2]],
    "kiwi 형태소": lambda s: [t.form for t in kiwi.tokenize(s) if t.tag in KEEP],
    "WordPiece":  lambda s: [t for t in wp.encode(s).tokens if t not in ("[CLS]", "[SEP]")],
    "tiktoken":   lambda s: [str(i) for i in o200k.encode(s)],
}
docs = [f"{q} {a}" for _, _, q, a in FAQ]
ids = [i for i, *_ in FAQ]
print(len(docs), "문서 /", len(TESTS), "평가 질문")

20 문서 / 12 평가 질문


## 12-2. 두 문장의 유사도를 손으로 — 토큰 세기 → 벡터 → 코사인·자카드

In [2]:
from collections import Counter
a, b = "택배 언제 와요", "배송은 언제 오나요"
for name in ["띄어쓰기", "kiwi 형태소", "글자 2-gram"]:
    ta, tb = Counter(TOKENIZERS[name](a)), Counter(TOKENIZERS[name](b))
    vocab = sorted(set(ta) | set(tb))
    va = np.array([ta[w] for w in vocab]); vb = np.array([tb[w] for w in vocab])
    cos = va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb))
    jac = len(set(ta) & set(tb)) / len(set(ta) | set(tb))
    print(f"[{name}] 어휘 {vocab}")
    print(f"   A {va.tolist()}  B {vb.tolist()}  → 코사인 {cos:.3f}  자카드 {jac:.3f}")

[띄어쓰기] 어휘 ['배송은', '언제', '오나요', '와요', '택배']
   A [0, 1, 0, 1, 1]  B [1, 1, 1, 0, 0]  → 코사인 0.333  자카드 0.200
[kiwi 형태소] 어휘 ['배송', '오', '택배']
   A [0, 1, 1]  B [1, 1, 0]  → 코사인 0.500  자카드 0.333
[글자 2-gram] 어휘 ['나요', '배송', '송은', '언제', '오나', '와요', '택배']
   A [0, 0, 0, 1, 0, 1, 1]  B [1, 1, 1, 1, 1, 0, 0]  → 코사인 0.258  자카드 0.143


## 12-3. TF-IDF에 내 토크나이저 끼우기

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

def build(name):
    vec = TfidfVectorizer(tokenizer=TOKENIZERS[name], lowercase=False, token_pattern=None)
    X = vec.fit_transform(docs)                       # 문서 20개 → [20, 어휘 수] 희소 행렬
    return vec, X

vec, X = build("kiwi 형태소")
print("행렬 모양:", X.shape, "| 0이 아닌 칸:", X.nnz)
print("어휘 앞 15개:", vec.get_feature_names_out()[:15].tolist())
q = vec.transform(["택배 언제 와요?"])
scores = (X @ q.T).toarray().ravel()                  # 행이 길이 1이라 내적 = 코사인
top = scores.argsort()[::-1][:3]
print("상위 3:", [(ids[i], round(float(scores[i]), 3), docs[i][:15]) for i in top])

행렬 모양: (20, 151) | 0이 아닌 칸: 245
어휘 앞 15개: ['1', '150', '2', '3', '30', '3000', '30000', '5', '6', '6000', '7', 'cm', 'kg', '가', '가능']
상위 3: [(101, 0.244, '배송은 언제 오나요? 주문 '), (702, 0.0, '쿠폰과 적립금을 같이 쓸 수'), (602, 0.0, '회원 탈퇴는 어떻게 하나요?')]


## 13-1. 토크나이저만 바꿔서 검색 품질 비교 — Hit@1·Hit@3

In [4]:
def evaluate(name, k=3):
    vec, X = build(name)
    h1 = h3 = 0
    fails = []
    for q, gold in TESTS:
        s = (X @ vec.transform([q]).T).toarray().ravel()
        got = [ids[i] for i in s.argsort()[::-1][:k]]
        h1 += got[0] == gold; h3 += gold in got
        if got[0] != gold:
            fails.append((q, gold, got[0]))
    return h1, h3, fails

results = {}
for name in TOKENIZERS:
    h1, h3, fails = evaluate(name)
    results[name] = fails
    print(f"{name:<12} Hit@1 {h1:>2}/{len(TESTS)}  Hit@3 {h3:>2}/{len(TESTS)}  어휘 {len(build(name)[0].vocabulary_)}")

띄어쓰기         Hit@1  3/12  Hit@3  5/12  어휘 201
글자 2-gram    Hit@1  7/12  Hit@3 11/12  어휘 329
kiwi 형태소     Hit@1  7/12  Hit@3  8/12  어휘 151
WordPiece    Hit@1  2/12  Hit@3  4/12  어휘 142
tiktoken     Hit@1  8/12  Hit@3 10/12  어휘 283


## 13-2. 틀린 질문 들여다보기

In [5]:
for name in ["띄어쓰기", "kiwi 형태소", "글자 2-gram"]:
    print(f"[{name}] 틀린 질문 {len(results[name])}개")
    for q, gold, got in results[name][:3]:
        print(f"   {q} → 정답 {gold} / 1등 {got} | 질문 토큰 {TOKENIZERS[name](q)}")

[띄어쓰기] 틀린 질문 9개
   택배 언제 와요? → 정답 101 / 1등 701 | 질문 토큰 ['택배', '언제', '와요?']
   내 택배 지금 어디쯤인지 보고 싶어요 → 정답 103 / 1등 702 | 질문 토큰 ['내', '택배', '지금', '어디쯤인지', '보고', '싶어요']
   미국으로 보내줄 수 있나요? → 정답 105 / 1등 502 | 질문 토큰 ['미국으로', '보내줄', '수', '있나요?']
[kiwi 형태소] 틀린 질문 5개
   내 택배 지금 어디쯤인지 보고 싶어요 → 정답 103 / 1등 702 | 질문 토큰 ['택배', '보']
   소파 같은 큰 물건은 어떻게 와요? → 정답 104 / 1등 101 | 질문 토큰 ['소파', '같', '크', '물건', '오']
   미국으로 보내줄 수 있나요? → 정답 105 / 1등 502 | 질문 토큰 ['미국', '보내', '수', '있']
[글자 2-gram] 틀린 질문 5개
   택배 언제 와요? → 정답 101 / 1등 301 | 질문 토큰 ['택배', '언제', '와요', '요?']
   내 택배 지금 어디쯤인지 보고 싶어요 → 정답 103 / 1등 501 | 질문 토큰 ['택배', '지금', '어디', '디쯤', '쯤인', '인지', '보고', '싶어', '어요']
   미국으로 보내줄 수 있나요? → 정답 105 / 1등 502 | 질문 토큰 ['미국', '국으', '으로', '보내', '내줄', '있나', '나요', '요?']


## 13-3. WordPiece가 약했던 이유 — 학습 말뭉치와 어휘 크기

In [6]:
from tokenizers import models, normalizers, pre_tokenizers, trainers
from sklearn.feature_extraction.text import TfidfVectorizer
corpus = open("shop_corpus.txt", encoding="utf-8").read().splitlines()

def train_wp(lines, vocab_size):
    t = Tokenizer(models.WordPiece(unk_token="[UNK]"))
    t.normalizer = normalizers.NFKC()
    t.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
    t.train_from_iterator(lines, trainers.WordPieceTrainer(vocab_size=vocab_size, min_frequency=2,
                          special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]))
    return t

for label, lines in [("말뭉치만", corpus), ("말뭉치+FAQ 문서", corpus + docs)]:
    for vs in [200, 800]:
        t = train_wp(lines, vs)
        f = lambda s, t=t: t.encode(s).tokens
        vec = TfidfVectorizer(tokenizer=f, lowercase=False, token_pattern=None)
        X = vec.fit_transform(docs)
        hit = sum(ids[(X @ vec.transform([q]).T).toarray().ravel().argmax()] == g for q, g in TESTS)
        unk = sum(tok == "[UNK]" for q, _ in TESTS for tok in f(q))
        print(f"{label:<12} vocab_size={vs:<4} 실제 어휘 {t.get_vocab_size():<4} Hit@1 {hit:>2}/12  질문 속 [UNK] {unk:>2}개  예: {f('택배 언제 와요?')}")

말뭉치만         vocab_size=200  실제 어휘 353  Hit@1  2/12  질문 속 [UNK] 21개  예: ['택', '##배', '언', '##제', '[UNK]', '?']
말뭉치만         vocab_size=800  실제 어휘 646  Hit@1  2/12  질문 속 [UNK] 21개  예: ['택배', '언제', '[UNK]', '?']
말뭉치+FAQ 문서   vocab_size=200  실제 어휘 480  Hit@1  9/12  질문 속 [UNK] 15개  예: ['택', '##배', '언', '##제', '[UNK]', '?']
말뭉치+FAQ 문서   vocab_size=800  실제 어휘 800  Hit@1  8/12  질문 속 [UNK] 15개  예: ['택배', '언제', '[UNK]', '?']
